# Extending AFINs

This notebook is a map for researchers who want to use this repo beyond the canned demo:

- write new Bayesian tasks within the supported factor families;
- restrict or expand the training distribution;
- add a new prior or likelihood family;
- change the design-matrix distribution used during amortized training.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "afin").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import importlib
import torch

import afin
import afin.api as _afin_api
import afin.spec as _afin_spec

importlib.reload(_afin_spec)
importlib.reload(_afin_api)
afin = importlib.reload(afin)
from afin import (
    BernoulliLogit,
    BinomialLogit,
    LinearGaussian,
    StudentTPrior,
)

_ = torch.manual_seed(0)

## 1. New tasks within the existing families

The public model builder supports the factor families the released AFINs were trained on: diagonal/full-rank Gaussian priors, Student-t and Laplace priors, and Gaussian, Bernoulli-logit, Binomial-logit, Student-t, and no-X Gaussian likelihoods.

If your task uses these families and stays inside the checkpoint range (`d <= 16`, `N <= 256`), you can usually try it directly with the pretrained model.

In [2]:
d = 3
n = 24
true_z = torch.tensor([0.5, -1.0, 0.8])
X = 0.7 * torch.randn(n, d) / d**0.5

y_reg = X @ true_z + 0.40 * torch.randn(n)
y_bin = torch.bernoulli(torch.sigmoid(X @ true_z))
y_count = torch.distributions.Binomial(total_count=8, logits=X @ true_z).sample()

prior = StudentTPrior(loc=0, scale=1, df=5)
observed = [
    LinearGaussian(design_matrix=X, sigma=0.40).observe(y_reg),
    BernoulliLogit(design_matrix=X).observe(y_bin),
    BinomialLogit(design_matrix=X, total_count=8).observe(y_count),
]

prior, observed

(StudentTPrior(loc=0, scale=1, df=5),
 [Observed(likelihood=LinearGaussian(design_matrix=tensor([[-0.4550, -0.4657, -0.1013],
          [-0.1754,  0.3430,  0.2797],
          [-0.1277, -0.8549,  0.1302],
          [-0.5106,  0.1414,  0.1245],
          [ 0.0484,  0.5002,  0.4513],
          [-0.0999, -0.5467, -0.6854],
          [ 0.2290,  0.3207,  0.2420],
          [-0.6285, -0.1380,  0.7489],
          [ 0.3032, -0.2366, -0.0701],
          [ 0.0742,  0.5615,  0.6411],
          [ 0.3824, -0.3410, -0.2480],
          [ 0.0128, -0.1991,  0.1004],
          [ 0.1777,  0.0454,  0.2590],
          [ 0.1783, -0.0413,  0.3203],
          [-0.1171,  0.0212,  0.2113],
          [ 0.9304, -0.5936, -0.6413],
          [-0.2720,  0.3528,  0.4265],
          [ 0.0719, -0.0931, -0.1583],
          [ 0.2196, -0.1597,  0.6099],
          [ 0.8414,  0.6898,  0.9620],
          [-0.4549, -0.1281, -0.4415],
          [-0.0344,  0.1324, -0.3074],
          [-0.6463,  0.0075, -0.3033],
          [ 0.07

## 2. Training on a different task distribution

The main CLI controls the amortized training distribution. Useful knobs:

```bash
.venv/bin/torchrun --standalone --nproc_per_node=1 -m afin.cli \
  --posterior-family flow \
  --d-min 1 --d-max 32 \
  --n-min 8 --n-max 512 \
  --prior-families diag_gaussian,diag_student_t \
  --likelihood-families gaussian,bernoulli_logit,student_t \
  --train-x-kappa-max 80 \
  --num-steps 200000 \
  --encoder-checkpoint --merge-checkpoint
```

Use narrower family lists for a specialized model; use wider `d/N` ranges and harder `X` settings when you want broader amortization.

## 3. Adding a new prior or likelihood family

Adding a new factor type is a real model change, not just a notebook change. The usual checklist is:

1. Add the user-facing distribution class in `src/afin/spec.py`.
2. Map that class to an internal family name in `_prior_meta(...)` or `_distribution_to_observation(...)`.
3. Add the family name to `PRIOR_FAMILIES` or `LIKELIHOOD_FAMILIES` in `src/afin/tasks.py`.
4. Add synthetic data generation for that family in `_sample_prior_family(...)` or `_sample_likelihood_batch(...)`.
5. Add its log density in `unnormalized_log_posterior(...)`.
6. Train a new checkpoint, because the factor-family ids and adapters have changed.

Sketch for a new likelihood:

```python
# src/afin/spec.py
@dataclass(frozen=True)
class PoissonLog(_Likelihood):
    design_matrix: Any

    def _to_observation(self, y, d):
        x = _matrix(self.design_matrix, d, "design_matrix")
        return _Observation(
            family="poisson_log",
            y=_site_vector(y, x.shape[0], "observation"),
            x=x,
        )
```

Then mirror that family inside `src/afin/tasks.py` for sampling and log probability.

## 4. Changing the design-matrix distribution

`src/afin/tasks.py` currently samples design matrices through `sample_design_matrix_batch(...)`, with modes such as iid, diagonal-scale, correlated, and Student-t designs. For research on covariate shift or harder inverse problems, edit:

- `X_FAMILIES`
- `sample_x_family_ids(...)`
- `sample_design_matrix_batch(...)`

The CLI flag `--train-x-kappa-max` already controls how ill-conditioned correlated designs can become. If you add a new `X` family, train and evaluate it explicitly rather than silently mixing it into the old benchmark.

## 5. A practical workflow

For a new research direction:

1. Prototype the Bayesian task with `GaussianPrior`, `LinearGaussian`, `BernoulliLogit`, etc.
2. Check whether a pretrained AFIN gives a reasonable posterior.
3. If the factor family or `d/N` range is out of distribution, train a specialized checkpoint.
4. Compare against NUTS on small tasks and report `m1`, `m2`, sliced W2, ESS, or downstream task loss.
5. Publish the checkpoint on Hugging Face and load it with `load_afin(path_to_snapshot / subfolder)`.
